# Step 1: Collect the data

This notebook downloads everything we need and saves it in the `data/` folder:

1. The **MLB schedule** for 2022–2026 from the MLB Stats API. This gives every game an ID (`game_pk`).
2. The **betting odds** for each game date from [SportsbookReview](https://www.sportsbookreview.com) (SBR):
   moneyline, run line and total, for every sportsbook on the page.

Each date's odds get saved to their own file in `data/raw_odds/`, so if the notebook gets stopped
you can just run it again and it picks up where it left off.

**The full scrape takes about 3 hours** (1,031 game dates × 3 pages, with a 2 second pause after each page).

In [ ]:
import os
import pandas as pd
from odds_helpers import get_mlb_schedule, scrape_date


## 1. MLB schedule

We get regular season and playoff games. The game types are:
`R` = regular season, `F` = wild card, `D` = division series, `L` = league championship, `W` = World Series.

In [7]:
SEASONS = [2022, 2023, 2024, 2025, 2026]

schedules = []
for season in SEASONS:
    print("Downloading schedule for", season)
    schedules.append(get_mlb_schedule(season))
schedule = pd.concat(schedules, ignore_index=True)

# Only keep games that were actually played (no postponed or cancelled games)
schedule = schedule[schedule["status"].isin(["Final", "Completed Early", "Game Over"])]

# Suspended games show up twice (when they started and when they finished), so keep the first one
schedule = schedule.drop_duplicates(subset="game_pk", keep="first")

os.makedirs("data", exist_ok=True)
schedule.to_csv("data/schedule.csv", index=False)

print(len(schedule), "games saved to data/schedule.csv")
schedule["season"].value_counts().sort_index()

12359 games saved to data/schedule.csv


season
2022    2470
2023    2471
2024    2472
2025    2477
2026    2469
Name: count, dtype: int64

In [8]:
schedule.head()

,game_pk,date,season,game_type,away,home,start_time,status
0,663178,2022-04-07,2022,R,Milwaukee Brewers,Chicago Cubs,2022-04-07T18:20:00Z,Final
1,662766,2022-04-07,2022,R,Cleveland Guardians,Kansas City Royals,2022-04-07T20:10:00Z,Final
2,662021,2022-04-07,2022,R,Pittsburgh Pirates,St. Louis Cardinals,2022-04-07T20:15:00Z,Final
3,662571,2022-04-07,2022,R,New York Mets,Washington Nationals,2022-04-07T23:05:00Z,Final
4,661577,2022-04-07,2022,R,Cincinnati Reds,Atlanta Braves,2022-04-08T00:08:00Z,Final


## 2. Scrape the odds from SBR

We only need to visit the dates that had games.

In [9]:
game_dates = sorted(schedule["date"].unique())
os.makedirs("data/raw_odds", exist_ok=True)

# Skip any dates we already downloaded
dates_to_scrape = []
for date in game_dates:
    if not os.path.exists(f"data/raw_odds/{date}.csv"):
        dates_to_scrape.append(date)

print(len(game_dates), "game dates in total,", len(dates_to_scrape), "still need to be scraped")
print("Estimated time:", round(len(dates_to_scrape) * 3 * 3.5 / 3600, 1), "hours")

1031 game dates in total, 1028 still need to be scraped
Estimated time: 3.0 hours


Set `TEST_RUN = True` to only scrape 3 dates and make sure everything works.
Set it to `False` for the full run.

In [10]:
TEST_RUN = False

if TEST_RUN:
    dates_to_scrape = dates_to_scrape[:3]

for i, date in enumerate(dates_to_scrape):
    print(f"{i + 1}/{len(dates_to_scrape)}  {date}")
    odds = scrape_date(date)
    odds.to_csv(f"data/raw_odds/{date}.csv", index=False)

print("Done!")

1/1028  2022-04-07
2/1028  2022-04-08
3/1028  2022-04-09
4/1028  2022-04-10
5/1028  2022-04-11
6/1028  2022-04-12
7/1028  2022-04-13
8/1028  2022-04-14
9/1028  2022-04-15
10/1028  2022-04-16
11/1028  2022-04-17
12/1028  2022-04-18
13/1028  2022-04-19
14/1028  2022-04-20
15/1028  2022-04-21
16/1028  2022-04-22
17/1028  2022-04-23
18/1028  2022-04-24
19/1028  2022-04-25
20/1028  2022-04-26
21/1028  2022-04-27
22/1028  2022-04-28
23/1028  2022-04-29
24/1028  2022-04-30
25/1028  2022-05-01
26/1028  2022-05-02
27/1028  2022-05-03
28/1028  2022-05-04
29/1028  2022-05-05
30/1028  2022-05-06
31/1028  2022-05-07
32/1028  2022-05-08
33/1028  2022-05-09
34/1028  2022-05-10
35/1028  2022-05-11
36/1028  2022-05-12
37/1028  2022-05-13
38/1028  2022-05-14
39/1028  2022-05-15
40/1028  2022-05-16
41/1028  2022-05-17
42/1028  2022-05-18
43/1028  2022-05-19
44/1028  2022-05-20
45/1028  2022-05-21
46/1028  2022-05-22
47/1028  2022-05-23
48/1028  2022-05-24
49/1028  2022-05-25
50/1028  2022-05-26
51/1028  

Here's what one day of raw odds looks like (one row per game, sportsbook and market):

In [11]:
example = pd.read_csv(f"data/raw_odds/{game_dates[0]}.csv")
example.head(10)

,date,market,sbr_game_id,start_time,away,home,book,open_away_odds,open_home_odds,open_home_spread,open_total,open_over_odds,open_under_odds,close_away_odds,close_home_odds,close_home_spread,close_total,close_over_odds,close_under_odds
0,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,betmgm,-150.0,125.0,NaN,NaN,NaN,NaN,-185.0,150.0,NaN,NaN,NaN,NaN
1,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,bet365,-160.0,140.0,NaN,NaN,NaN,NaN,-180.0,160.0,NaN,NaN,NaN,NaN
2,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,draftkings,-150.0,130.0,NaN,NaN,NaN,NaN,-180.0,155.0,NaN,NaN,NaN,NaN
3,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,fanduel,-154.0,130.0,NaN,NaN,NaN,NaN,-190.0,160.0,NaN,NaN,NaN,NaN
4,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,bet_rivers_co,-148.0,125.0,NaN,NaN,NaN,NaN,-177.0,148.0,NaN,NaN,NaN,NaN
5,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,caesars,-145.0,125.0,NaN,NaN,NaN,NaN,-179.0,160.0,NaN,NaN,NaN,NaN
6,2022-04-07,moneyline,264308,2022-04-07T20:10:00+00:00,Cleveland Guardians,Kansas City Royals,betmgm,-115.0,-105.0,NaN,NaN,NaN,NaN,-130.0,105.0,NaN,NaN,NaN,NaN
7,2022-04-07,moneyline,264308,2022-04-07T20:10:00+00:00,Cleveland Guardians,Kansas City Royals,bet365,-125.0,105.0,NaN,NaN,NaN,NaN,-125.0,105.0,NaN,NaN,NaN,NaN
8,2022-04-07,moneyline,264308,2022-04-07T20:10:00+00:00,Cleveland Guardians,Kansas City Royals,draftkings,-120.0,100.0,NaN,NaN,NaN,NaN,-130.0,110.0,NaN,NaN,NaN,NaN
9,2022-04-07,moneyline,264308,2022-04-07T20:10:00+00:00,Cleveland Guardians,Kansas City Royals,fanduel,-108.0,-108.0,NaN,NaN,NaN,NaN,-124.0,106.0,NaN,NaN,NaN,NaN
